# Yahoo OHLCV audit

This notebook audits the forecasting data currently stored in the repository and re-downloads daily OHLCV data from Yahoo Finance for the same 23 assets.

The goal is to verify which preprocessing techniques from the financial data preprocessing workshop can be applied to this dataset.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "util.py").exists():
    PROJECT_ROOT = next(p for p in PROJECT_ROOT.parents if (p / "util.py").exists())

sys.path.insert(0, str(PROJECT_ROOT / "model" / "preprocessing"))

import pandas as pd

from preprocessing_utils import (
    DATA_OUT,
    get_forecasting_tickers,
    get_forecasting_date_range,
    download_yahoo_ohlcv,
    extract_field,
    compute_universe_activity,
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_OUT:", DATA_OUT)


PROJECT_ROOT: /Users/jchulvi/projects/Neural-Networks-Forecasting
DATA_OUT: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing


## Existing repository data

The existing forecasting data is based on adjusted close prices and log returns.

In [2]:
prices = pd.read_parquet(PROJECT_ROOT / "data" / "precios_close.parquet")
returns = pd.read_parquet(PROJECT_ROOT / "data" / "returns.parquet")

audit_rows = [
    {
        "dataset": "precios_close",
        "shape": str(prices.shape),
        "start_date": prices.index.min(),
        "end_date": prices.index.max(),
        "n_assets": prices.shape[1],
        "columns_available": "Close only",
    },
    {
        "dataset": "returns",
        "shape": str(returns.shape),
        "start_date": returns.index.min(),
        "end_date": returns.index.max(),
        "n_assets": returns.shape[1],
        "columns_available": "Log returns only",
    },
]

audit = pd.DataFrame(audit_rows)
display(audit)

print("Tickers:")
print(list(prices.columns))


,dataset,shape,start_date,end_date,n_assets,columns_available
0,precios_close,"(16196, 23)",1962-01-02,2026-05-08,23,Close only
1,returns,"(16195, 23)",1962-01-03,2026-05-08,23,Log returns only


Tickers:
['AEP', 'BA', 'CAT', 'CNP', 'CVX', 'DIS', 'DTE', 'ED', 'GD', 'GE', 'HON', 'HPQ', 'IBM', 'IP', 'JNJ', 'KO', 'KR', 'MMM', 'MO', 'MRK', 'MSI', 'PG', 'XOM']


## Re-download Yahoo OHLCV

Yahoo Finance provides daily Open, High, Low, Close and Volume. This is enough to build daily count, volume and dollar bars, but not transaction-level tick bars.

In [3]:
tickers = get_forecasting_tickers()
start, end = get_forecasting_date_range()

# yfinance treats end as exclusive, so add one day.
start_str = start.strftime("%Y-%m-%d")
end_str = (end + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

print("Downloading:", len(tickers), "tickers")
print("Range:", start_str, "->", end_str)

ohlcv_path = DATA_OUT / "yahoo_ohlcv.parquet"

ohlcv = download_yahoo_ohlcv(
    tickers=tickers,
    start=start_str,
    end=end_str,
    output_path=ohlcv_path,
)

print("Saved:", ohlcv_path)
print("Shape:", ohlcv.shape)
display(ohlcv.head())


Downloading: 23 tickers
Range: 1962-01-02 -> 2026-05-09


Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/yahoo_ohlcv.parquet
Shape: (16196, 115)


,Close__AEP,Close__BA,Close__CAT,Close__CNP,Close__CVX,Close__DIS,Close__DTE,Close__ED,Close__GD,Close__GE,...,Volume__IP,Volume__JNJ,Volume__KO,Volume__KR,Volume__MMM,Volume__MO,Volume__MRK,Volume__MSI,Volume__PG,Volume__XOM
Date,,,,,,,,,,,,,,,,,,,,,
1962-01-02,0.887649,0.190931,0.462333,0.284116,0.307970,0.057055,0.386574,0.238809,0.168170,0.617245,...,51552,0,806400,153600,254509,345600,633830,65671,192000,902400
1962-01-03,0.886033,0.194749,0.466836,0.281340,0.307275,0.057821,0.383392,0.238809,0.173799,0.611052,...,53736,345600,1574400,131200,505190,1209600,6564672,77611,428800,1200000
1962-01-04,0.873098,0.192840,0.478844,0.281340,0.304494,0.057821,0.380211,0.238072,0.174503,0.603827,...,48494,216000,844800,99200,254509,2592000,1199750,59701,326400,1088000
1962-01-05,0.853696,0.189022,0.483348,0.274553,0.296847,0.058012,0.372256,0.232912,0.175207,0.588344,...,76891,129600,1420800,182400,376979,2937600,520646,107462,544000,1222400
1962-01-08,0.847229,0.189499,0.486350,0.271468,0.295456,0.057821,0.373052,0.234018,0.178021,0.587311,...,93929,172800,2035200,208000,399942,1382400,1380845,89551,1523200,1388800


## OHLCV availability check

In [4]:
close = extract_field(ohlcv, "Close")
volume = extract_field(ohlcv, "Volume")
activity = compute_universe_activity(ohlcv)

availability = pd.DataFrame({
    "field": ["Close", "Volume"],
    "shape": [str(close.shape), str(volume.shape)],
    "missing_values": [int(close.isna().sum().sum()), int(volume.isna().sum().sum())],
    "start_date": [close.index.min(), volume.index.min()],
    "end_date": [close.index.max(), volume.index.max()],
})

availability_path = DATA_OUT / "yahoo_ohlcv_audit_summary.csv"
availability.to_csv(availability_path, index=False)

activity_path = DATA_OUT / "universe_daily_activity.csv"
activity.to_csv(activity_path)

print("Saved:", availability_path)
print("Saved:", activity_path)
display(availability)
display(activity.head())


Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/yahoo_ohlcv_audit_summary.csv
Saved: /Users/jchulvi/projects/Neural-Networks-Forecasting/data/preprocessing/universe_daily_activity.csv


,field,shape,missing_values,start_date,end_date
0,Close,"(16196, 23)",0,1962-01-02,2026-05-08
1,Volume,"(16196, 23)",0,1962-01-02,2026-05-08


,total_volume,total_dollar,valid_close_assets,valid_volume_assets
Date,,,,
1962-01-02,10928249,2.112313e+06,23,23
1962-01-03,17247261,2.500944e+06,23,23
1962-01-04,13539399,2.201601e+06,23,23
1962-01-05,12668601,2.264614e+06,23,23
1962-01-08,14676256,2.919913e+06,23,23


## Conclusion

The saved forecasting files only contain close prices and returns. After re-downloading daily OHLCV from Yahoo Finance, we can construct daily count bars, volume bars and dollar bars using the aggregated universe activity.

Real tick bars cannot be constructed because Yahoo Finance does not provide transaction-level trades in this dataset.